In [12]:
print('Radha!')


Radha!


In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from dotenv import  load_dotenv


load_dotenv()

True

In [14]:
# 2. Define Tools
@tool
def get_customer(customer_id: int) -> dict:
    """Get customer information."""
    return {"customer_id": customer_id, "name": "Rahul", "plan": "Premium"}

@tool
def delete_customer(customer_id: int) -> str:
    """Delete a customer account."""
    return f"Customer {customer_id} deleted."

In [15]:
# 3. Define Security Middleware
@wrap_tool_call
def security_middleware(request, handler):
    tool_name = request.tool_call["name"]
    print(f"\n[MIDDLEWARE] Intercepted tool call: '{tool_name}'")

    if tool_name == "delete_customer":
        print("[MIDDLEWARE] 🛡️ BLOCKED: Dangerous action detected!")
        return ToolMessage(
            content="BLOCKED: Deleting customers is not allowed.",
            tool_call_id=request.tool_call["id"],
        )

    print("[MIDDLEWARE] ✅ ALLOWED: Executing tool...")
    return handler(request)


In [16]:
# 4. Create Agent
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(
    model=llm,
    tools=[get_customer, delete_customer],
    middleware=[security_middleware],
)

In [17]:
# 5. Run & Print Response
print("--- Running Test Request ---")
result = agent.invoke(
        {"messages": [{"role": "user", "content": "Delete customer 101."}]}
    )

print("\n--- Final Output ---")
print(result["messages"][-1].content)

--- Running Test Request ---

[MIDDLEWARE] Intercepted tool call: 'delete_customer'
[MIDDLEWARE] 🛡️ BLOCKED: Dangerous action detected!

--- Final Output ---
I'm unable to delete customer accounts. If you need assistance with something else, please let me know!
